# 8장. 텍스트 정제: 기준을 설계하기

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 2장, 7장 |
| 이 장의 결과물 | `clean_en(text)`, `clean_ko(text)`, 리포트용 "정제 기준과 근거" 표 |
| 관련 학습 목표 | ⑤ 전처리 기준(특수문자 범위)을 설계한 근거 |

**학습 목표**
- 정제가 왜 필요한지, 너무 많이 지우면 무엇을 잃는지 설명할 수 있다.
- 정규식으로 URL, 이메일, 숫자, 특수문자, 연속 공백을 다룰 수 있다.
- "지울 것을 고르는 방식"과 "남길 것을 고르는 방식"의 차이를 안다.
- 애매한 사례(`C++`, `3D`, `don't`, `北`)에 대해 기준을 정하고 근거를 쓸 수 있다.
- 정제 전후를 숫자로 비교해 기준이 효과가 있는지 확인할 수 있다.

> 과제는 전처리 기준을 **학습자가 직접 설계하고 근거를 리포트에 쓰라**고 요구합니다. 이 장은 정답 규칙을 외우는 장이 아니라, **판단하는 방법**을 익히는 장입니다. 아래에서 제시하는 규칙은 하나의 합리적인 선택일 뿐이고, 근거를 댈 수 있다면 다르게 정해도 됩니다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

from collections import Counter

en = load_english()
ko = load_korean()
print("영어:", en.shape, "| 한국어:", ko.shape)

영어: (4797, 2) | 한국어: (7000, 3)


## 8.1 정제의 목적

BoW는 **글자가 정확히 같아야** 같은 단어로 봅니다. 아래 표현들은 사람 눈에는 모두 "space"지만 컴퓨터에는 서로 다른 단어입니다.

In [2]:
raw_forms = Counter()
for text in en["text"]:
    for tok in text.split():
        if re.sub(r"[^a-z]", "", tok.lower()) == "space":
            raw_forms[tok] += 1

print(f"공백으로만 자르면 'space'가 {len(raw_forms)}가지 모양으로 나뉜다:")
print(raw_forms.most_common(15))

공백으로만 자르면 'space'가 43가지 모양으로 나뉜다:
[('space', 566), ('Space', 407), ('SPACE', 98), ('space,', 54), ('space.', 46), ('Space:', 34), ('"Space', 15), ('SPACE:', 10), ('space?', 8), ('Space",', 6), ('SPACE,', 4), ('space)', 4), ('(Space', 4), ('SPACE.', 3), ('SPACE:1999', 3)]


같은 단어가 여러 열로 쪼개지면
- 단어 빈도와 문서 빈도가 나뉘어 **IDF가 실제보다 크게** 계산되고,
- 검색어 `space`가 `space.`나 `Space`만 가진 문서를 **찾지 못하고**,
- 어휘(벡터 차원)가 쓸데없이 커집니다.

정제의 목적은 두 가지입니다.
1. **같은 의미의 다른 표기를 하나로 합치기** (대소문자, 붙은 문장부호)
2. **의미 없는 잡음 지우기** (특수문자, 자리표시자, 인코딩된 파일 조각)

반대로 **지나치게 지우면 정보를 잃습니다.** `C++`에서 `++`를 지우면 `c`만 남고, 한자 `北`을 지우면 북한 기사의 강한 단서가 사라집니다. 정제는 이 둘 사이의 균형을 정하는 일입니다.

## 8.2 소문자 변환

영어는 `NASA`, `Nasa`, `nasa`를 하나로 합치기 위해 소문자로 바꿉니다. 대가로 고유명사와 일반명사가 섞일 수 있습니다(`Apple` 회사 vs `apple` 과일). 주제 분류에서는 합치는 이득이 훨씬 큽니다.

한국어에는 대소문자가 없으므로 영향이 없습니다. 다만 한국어 제목에도 `AI`, `SKT`, `LG` 같은 영문이 자주 섞이므로, 영문 토큰은 소문자로 맞춥니다.

In [3]:
print("영문이 섞인 한국어 제목 비율:", f"{ko['text'].str.contains('[A-Za-z]').mean():.1%}")
print(ko.loc[ko["text"].str.contains("[A-Za-z]"), "text"].head(5).tolist())

영문이 섞인 한국어 제목 비율: 25.8%
['KT 데이터 공유 앱 Y데이터박스 가입자 10만 돌파', '수괴 사망에도 IS 위협 여전…충성맹세에 말리 테러 배후 자처종합', '애플 25일 스페셜 이벤트…TV·뉴스·아이패드 새 버전 선보일듯', 'MBC 조선일보 정조준…스트레이트서 검찰유착 주장', '강아정 3점 버저비터 여자농구 KB 삼성생명에 역전극']


## 8.3 정규식 기본

정규식(regular expression)은 "글자 패턴"을 표현하는 작은 언어입니다. 파이썬에서는 `re` 모듈을 씁니다.

| 패턴 | 뜻 | 예 |
|---|---|---|
| `[abc]` | a, b, c 중 한 글자 | |
| `[a-z]` | 소문자 한 글자 (범위) | |
| `[^a-z]` | 소문자가 **아닌** 한 글자 (`^`가 대괄호 안 맨 앞이면 부정) | |
| `\d` / `\s` / `\w` | 숫자 / 공백류(스페이스, 탭, 줄바꿈) / 단어 글자 | |
| `+` / `*` / `?` | 1번 이상 / 0번 이상 / 0 또는 1번 | `\d+` = 숫자 연속 |
| `{2,4}` | 2~4번 | `\d{2,4}` |
| `(?:...)` | 묶기 (결과로 따로 뽑지 않음) | `(?:'[a-z]+)*` |
| `\b` | 단어 경계 | |

| 함수 | 하는 일 |
|---|---|
| `re.sub(패턴, 바꿀 글자, 텍스트)` | 패턴에 맞는 부분을 모두 바꾼다 |
| `re.findall(패턴, 텍스트)` | 패턴에 맞는 부분을 모두 리스트로 뽑는다 |
| `re.compile(패턴)` | 패턴을 미리 만들어 둔다 (여러 번 쓸 때 빠름) |

> **`\w`의 함정**: 파이썬의 `\w`는 유니코드 글자를 모두 포함해서 한글, 한자, 숫자, 밑줄(`_`)까지 "단어 글자"로 봅니다. 영어만 남기려고 `\w`를 쓰면 의도와 다르게 동작할 수 있으므로 `[a-z]`처럼 범위를 직접 씁니다.

In [4]:
sample = "Check http://www.nasa.gov/shuttle  or mail <email> ASAP!!! Launch: 1993-04-05, cost $1.5B."

print("URL 지우기    :", re.sub(r"(https?://|www\.)\S+", " ", sample))
print("숫자만 뽑기   :", re.findall(r"\d+", sample))
print("영문자만 남기기:", re.sub(r"[^a-z\s]", " ", sample.lower()))
print("연속 공백 정리:", " ".join(re.sub(r"[^a-z\s]", " ", sample.lower()).split()))
print("\\w 확인       :", re.findall(r"\w+", "space_shuttle 우주 北 3D"))

URL 지우기    : Check    or mail <email> ASAP!!! Launch: 1993-04-05, cost $1.5B.
숫자만 뽑기   : ['1993', '04', '05', '1', '5']
영문자만 남기기: check http   www nasa gov shuttle  or mail  email  asap    launch              cost     b 
연속 공백 정리: check http www nasa gov shuttle or mail email asap launch cost b
\w 확인       : ['space_shuttle', '우주', '北', '3D']


### 두 가지 접근: 지울 것을 고르기 vs 남길 것을 고르기

| 방식 | 코드 | 특징 |
|---|---|---|
| **블랙리스트** (지울 것을 나열) | `re.sub(r"[!?.,;:()]", " ", text)` | 목록에 없는 이상한 문자(`…`, `·`, 이모지, 제어 문자)가 그대로 남는다 |
| **화이트리스트** (남길 것을 나열) | `re.sub(r"[^a-z\s]", " ", text)` 또는 `re.findall(r"[a-z]+", text)` | 예상하지 못한 문자도 모두 지워진다. 남길 범위를 명확히 설명할 수 있다 |

실제 데이터에는 예상하지 못한 문자가 반드시 섞여 있으므로 **화이트리스트 방식**이 안전합니다. 리포트에도 "무엇을 지웠는가"보다 "무엇을 남겼는가"로 쓰면 기준이 분명해집니다.

In [5]:
weird = Counter(ch for t in ko["text"] for ch in t if not re.match(r"[0-9a-zA-Z가-힣\s]", ch))
print("한국어 제목에서 한글·영문·숫자·공백이 아닌 문자 상위 20개:")
print(weird.most_common(20))

한국어 제목에서 한글·영문·숫자·공백이 아닌 문자 상위 20개:
[('…', 2968), ('·', 1137), ('.', 646), ('%', 244), ('美', 201), ('北', 188), ('中', 119), ('朴', 97), ('靑', 70), ('↑', 67), ('日', 67), ('∼', 65), ('與', 46), ('英', 35), ('野', 31), ('文', 30), ('㎜', 21), ('ㆍ', 21), ('↓', 21), ('伊', 21)]


`…`, `·`, `↑` 같은 문자와 함께 **한자**가 많이 보입니다. 블랙리스트로는 이런 문자를 미리 다 나열하기 어렵습니다. 한자는 8.4절에서 따로 판단합니다.

## 8.4 한국어 정제와 한자

한국어 정제는 보통 "한글, 영문, 숫자, 공백만 남긴다"(`[^가-힣a-zA-Z0-9\s]` 제거)로 시작합니다. 그런데 뉴스 제목에는 한자 한 글자 약어가 자주 쓰입니다.

In [6]:
HANJA = re.compile(r"[一-鿿]")
has_hanja = ko["text"].str.contains(HANJA)
print(f"한자가 들어간 제목: {has_hanja.mean():.1%}")

hanja_counts = Counter(ch for t in ko["text"] for ch in HANJA.findall(t))
print("자주 나오는 한자:", hanja_counts.most_common(12))

print("\n한자가 있는 제목의 주제 분포:")
print(ko.loc[has_hanja, "label"].value_counts().to_string())

한자가 들어간 제목: 13.1%
자주 나오는 한자: [('美', 201), ('北', 188), ('中', 119), ('朴', 97), ('靑', 70), ('日', 67), ('與', 46), ('英', 35), ('野', 31), ('文', 30), ('伊', 21), ('韓', 19)]

한자가 있는 제목의 주제 분포:
label
정치      425
세계      335
사회       54
경제       41
스포츠      25
IT과학     24
생활문화     15


`北`(북한), `美`(미국), `中`(중국), `日`(일본), `與`(여당), `野`(야당) 같은 약어가 **정치·세계 주제에 몰려** 있습니다. 한글만 남기면 "北 미사일 발사"가 "미사일 발사"가 되어 가장 강한 단서를 잃습니다. 그래서 한자 범위(`一-鿿`, 유니코드 CJK 통합 한자)도 남기기로 합니다.

In [7]:
PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")      # 7장 mask_pii가 넣은 자리표시자
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")


def clean_ko(text: str) -> str:
    """한글·한자·영문·숫자·공백만 남기고 연속 공백을 하나로 만든다."""
    text = PLACEHOLDER.sub(" ", text)
    return " ".join(KO_CLEAN.sub(" ", text).split())


for t in ko.loc[has_hanja, "text"].head(5):
    print(f"{t}\n  → {clean_ko(t)}")

朴대통령 지방·중앙이 힘모아 위기를 기회로
  → 朴대통령 지방 중앙이 힘모아 위기를 기회로
강기정 曺청문회 일정에 8월30일 법정시한 지키지 않아 유감
  → 강기정 曺청문회 일정에 8월30일 법정시한 지키지 않아 유감
北 고려항공 中 칭다오 운항…관광객 수송 목적
  → 北 고려항공 中 칭다오 운항 관광객 수송 목적
국민통합 시대 여행업계 韓中 관계개선으로 유커 돌아오도록 해야
  → 국민통합 시대 여행업계 韓中 관계개선으로 유커 돌아오도록 해야
朴대통령여야대표 회동에 안보실장·외교·통일장관 첫 배석
  → 朴대통령여야대표 회동에 안보실장 외교 통일장관 첫 배석


숫자는 정제 단계에서는 남겨 둡니다. "3분기", "5G"처럼 글자와 붙은 숫자가 있어서, 여기서 지우면 형태소 분석기가 단어를 엉뚱하게 자르기 때문입니다. **숫자로만 된 토큰은 토큰화한 뒤에** 지웁니다(10장).

## 8.5 영어 정제: 애매한 사례 판단하기

영어 규칙의 후보로 "알파벳과 단어 안의 아포스트로피만 남긴다"를 정해 보고, 애매한 사례에 어떤 결과가 나오는지 확인합니다. 화이트리스트를 `re.findall`로 구현합니다.

- 패턴 `[a-z]+(?:'[a-z]+)*`: 알파벳 덩어리, 그 뒤에 `'알파벳` 덩어리가 0번 이상 → `don't`, `nasa's`는 한 토큰
- 둥근 따옴표 `’`는 일반 아포스트로피 `'`로 먼저 바꿉니다(같은 단어가 둘로 나뉘지 않게).

In [8]:
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")


def clean_en(text: str) -> str:
    """소문자로 바꾸고, 알파벳과 단어 안의 아포스트로피만 남긴다."""
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


cases = ["C++", "3D graphics", "windows95", "X11R5", "e-mail", "don't", "NASA's", "NASA’s",
         "U.S.", "1993", "$1.5B", "'quoted'", "rock'n'roll"]
for c in cases:
    print(f"{c:<14} → {clean_en(c)!r}")

C++            → 'c'
3D graphics    → 'd graphics'
windows95      → 'windows'
X11R5          → 'x r'
e-mail         → 'e mail'
don't          → "don't"
NASA's         → "nasa's"
NASA’s         → "nasa's"
U.S.           → 'u s'
1993           → ''
$1.5B          → 'b'
'quoted'       → 'quoted'
rock'n'roll    → "rock'n'roll"


결과를 보고 사례마다 판단합니다.

| 사례 | 결과 | 판단 | 근거 |
|---|---|---|---|
| `NASA` → `nasa` | 합쳐짐 | ✅ 수용 | 대소문자 차이를 없애는 것이 목적 |
| `don't`, `nasa's` | 한 토큰 유지 | ✅ 수용 | 아포스트로피를 지우면 `don`+`t`, `nasa`+`s` 같은 조각이 생긴다. `'s`는 10장에서 따로 뗀다 |
| `NASA’s` | `nasa's` | ✅ 수용 | 둥근 따옴표를 통일하지 않으면 같은 단어가 둘로 나뉜다 |
| `1993`, `$1.5B` | 사라짐 | ✅ 수용 | 연도, 금액, 버전, 점수는 문서마다 값이 달라서 주제 신호가 약하고 어휘만 키운다 |
| `3D` → `d`, `X11R5` → `x r` | 조각 남음 | ⚠️ 감수 | 숫자가 섞인 용어가 깨진다. 남는 한 글자 조각은 10장의 길이 필터(2글자 미만 제거)로 지운다 |
| `C++` → `c` | 정보 손실 | ⚠️ 감수 | 프로그래밍 언어 이름을 잃지만, 이 데이터에서 `c++`이 주제를 가르는 핵심어는 아니다 |
| `e-mail` → `e mail` | 쪼개짐 | ⚠️ 감수 | `mail`은 살아남는다 |

⚠️ 항목은 **알고도 받아들인 손실**입니다. 리포트에 이렇게 "무엇을 잃는지 알고 선택했다"고 쓰는 것이 좋은 근거입니다. 다르게 판단해도 됩니다. 예를 들어 `comp.graphics`의 `3d`를 살리고 싶다면 숫자+영문 조합을 남기는 규칙을 추가하고, 그 효과를 8.6절 방식으로 측정하면 됩니다.

## 8.6 정제 효과를 숫자로 확인하기

규칙을 정했으면 실제 데이터에서 **정말 효과가 있었는지** 확인합니다.

In [9]:
def vocab_stats(token_lists) -> dict:
    counts = Counter(t for toks in token_lists for t in toks)
    return {"어휘 수": len(counts),
            "한 번만 나온 단어 비율": f"{np.mean([c == 1 for c in counts.values()]):.1%}",
            "문서당 토큰 수": round(np.mean([len(t) for t in token_lists]), 1)}


en_raw_tokens = [t.split() for t in en["text"]]
en_clean_tokens = [clean_en(t).split() for t in en["text"]]
ko_raw_tokens = [t.split() for t in ko["text"]]
ko_clean_tokens = [clean_ko(t).split() for t in ko["text"]]

pd.DataFrame({
    "영어 정제 전 (공백 split)": vocab_stats(en_raw_tokens),
    "영어 정제 후": vocab_stats(en_clean_tokens),
    "한국어 정제 전": vocab_stats(ko_raw_tokens),
    "한국어 정제 후": vocab_stats(ko_clean_tokens),
})

,영어 정제 전 (공백 split),영어 정제 후,한국어 정제 전,한국어 정제 후
어휘 수,105324,36444,23946,22729
한 번만 나온 단어 비율,57.0%,41.5%,77.1%,71.6%
문서당 토큰 수,187.4,178.4,6.5,7.2


영어는 정제만으로 어휘가 크게 줄어듭니다. 앞에서 본 `space.`, `Space` 같은 변형이 합쳐졌기 때문입니다. 한국어는 정제 효과가 작습니다. 한국어의 진짜 문제는 문장부호가 아니라 **조사와 어미**이고, 이것은 9장 토큰화에서 다룹니다.

정제 전후 샘플도 눈으로 확인합니다. 숫자만 보면 놓치는 이상한 결과를 찾을 수 있습니다.

In [10]:
rng = np.random.default_rng(0)
for i in rng.choice(len(en), 2, replace=False):
    print(f"[{en.loc[i, 'label']}] 원문 앞 200자:\n{en.loc[i, 'text'][:200]!r}")
    print(f"정제 후 앞 200자:\n{clean_en(en.loc[i, 'text'])[:200]!r}\n")

[comp.graphics] 원문 앞 200자:
"I've got a 386 20Hz computer which is under warranty and my Trident\n8900C video card is starting to play-up (surprise, surprise). Therefore\nI'm going to try to exchange it for a better card.\n\nThe BIG "
정제 후 앞 200자:
"i've got a hz computer which is under warranty and my trident c video card is starting to play up surprise surprise therefore i'm going to try to exchange it for a better card the big question is whic"

[rec.sport.hockey] 원문 앞 200자:
'This is the price hockey fans have to pay for the mistake of leaving\nESPN 5 or 6 years ago...it will take time to correct for that mistake.\n\nBut returning to ESPN was the correct decision last year...'
정제 후 앞 200자:
'this is the price hockey fans have to pay for the mistake of leaving espn or years ago it will take time to correct for that mistake but returning to espn was the correct decision last year by next ti'



### 📝 리포트용 정리: 정제 기준과 근거

리포트의 "전처리 파이프라인" 부분에 아래처럼 정리합니다. 각 줄에 **규칙 + 이유 + 감수한 손실**을 함께 씁니다.

| 항목 | 영어 | 한국어 | 근거 |
|---|---|---|---|
| 남기는 문자 | 알파벳, 단어 안 아포스트로피 | 한글, 한자, 영문, 숫자, 공백 | 화이트리스트 방식으로 예상 못한 문자까지 제거 |
| 대소문자 | 소문자로 통일 | 영문만 소문자로 | 같은 단어의 표기 차이 제거 |
| 숫자 | 제거 | 정제 단계는 유지, 숫자만 있는 토큰은 토큰화 후 제거 | 영어: 연도·버전·점수는 주제 신호가 약함 / 한국어: "3분기", "5G" 같은 결합어 보존 |
| 한자 | 해당 없음 | 유지 | 제목의 약 13%에 있고 `北`, `美`, `與` 등이 정치·세계 주제의 강한 신호 |
| 자리표시자 | `<email>` 등 제거 | 같음 | 7장에서 가린 개인정보가 단어로 들어가지 않게 |
| 감수한 손실 | `C++`→`c`, `3D`→`d` | 특수기호 의미(`↑` 상승 등) | 해당 용어가 주제의 핵심어가 아님 |

## ✍️ 실습: 정제 함수 직접 만들기

아래 테스트를 모두 통과하는 `my_clean_en(text)`를 작성하세요. 위의 `clean_en`을 보지 않고 직접 써 보는 것이 좋습니다.

In [11]:
TESTS = [
    ("NASA's Shuttle!!", "nasa's shuttle"),
    ("Visit http://nasa.gov now", "visit now"),
    ("It costs $1.5B in 1993.", "it costs b in"),
    ("Don’t   panic", "don't panic"),
    ("<email> wrote: hi", "wrote hi"),
    ("", ""),
]


def run_tests(func):
    for given, expected in TESTS:
        got = func(given)
        print(("✅" if got == expected else "❌"), repr(given), "→", repr(got),
              "" if got == expected else f"(기대: {expected!r})")

In [12]:
# 여기에 작성하세요

**정답**

In [13]:
URL = re.compile(r"(https?://|www\.)\S+")


def my_clean_en(text: str) -> str:
    text = URL.sub(" ", text)                              # URL을 먼저 지운다 (안 그러면 http, nasa, gov가 단어로 남는다)
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(re.findall(r"[a-z]+(?:'[a-z]+)*", text))


run_tests(my_clean_en)

✅ "NASA's Shuttle!!" → "nasa's shuttle" 
✅ 'Visit http://nasa.gov now' → 'visit now' 
✅ 'It costs $1.5B in 1993.' → 'it costs b in' 
✅ 'Don’t   panic' → "don't panic" 
✅ '<email> wrote: hi' → 'wrote hi' 
✅ '' → '' 


두 번째 테스트처럼 **URL은 정제 전에 통째로 지워야** 합니다. 그렇지 않으면 `http`, `nasa`, `gov`가 단어로 남습니다. 7장의 `mask_pii`가 불러오는 단계에서 URL을 `<url>`로 바꿔 두므로, 실제 파이프라인에서는 자리표시자만 지우면 됩니다. **처리 순서**도 기준의 일부입니다.

## 🧐 퀴즈

**Q1.** 특수문자를 지울 때 "지울 문자 목록"보다 "남길 문자 범위"를 정하는 방식이 안전한 이유는?
<details><summary>▶ 정답 보기</summary>

실제 데이터에는 `…`, `·`, 제어 문자, 이모지처럼 미리 예상하지 못한 문자가 섞여 있어서 지울 목록을 완전하게 만들 수 없기 때문입니다. 남길 범위를 정하면 그 밖의 모든 문자가 자동으로 지워지고, 리포트에도 기준을 명확히 쓸 수 있습니다.
</details>

**Q2.** 한국어 뉴스 제목에서 한자를 지우면 어떤 문제가 생기나요?
<details><summary>▶ 정답 보기</summary>

`北`(북한), `美`(미국), `與`(여당) 같은 한 글자 한자 약어가 정치·세계 주제의 강한 단서인데, 이것이 사라져 분류와 검색 성능이 떨어집니다. 이 데이터에서는 제목의 약 13%에 한자가 있습니다.
</details>

**Q3.** 영어에서 숫자를 지우기로 한 근거와, 그 대가는?
<details><summary>▶ 정답 보기</summary>

근거: 연도, 버전, 금액, 점수처럼 숫자 값은 문서마다 달라서 주제를 가르는 신호가 약하고 어휘만 키웁니다. 대가: `3D`→`d`, `X11R5`→`x r`처럼 숫자가 섞인 용어가 깨집니다.
</details>

**Q4.** `\w+`로 영어 단어만 뽑으려 하면 생길 수 있는 문제는?
<details><summary>▶ 정답 보기</summary>

파이썬의 `\w`는 유니코드 글자를 모두 포함해 숫자, 밑줄, 한글, 한자도 단어 글자로 봅니다. 영어만 남기려면 `[a-z]`처럼 범위를 직접 지정해야 합니다.
</details>

## ✅ 체크포인트
- [ ] `re.sub`, `re.findall`로 패턴을 지우고 뽑을 수 있다.
- [ ] 화이트리스트 방식으로 영어·한국어 정제 함수를 만들 수 있다.
- [ ] 애매한 사례 표를 만들고, 받아들인 손실까지 근거로 쓸 수 있다.
- [ ] 정제 전후의 어휘 수 변화를 측정할 수 있다.

## 🔗 과제 연결과 다음 장
- `clean_en`, `clean_ko`는 `src/preprocess.py`의 정제 단계가 됩니다. "정제 기준과 근거" 표는 README의 전처리 설계 근거에 씁니다(학습 목표 ⑤).
- **9장. 토큰화**에서 정제된 문장을 단어로 자르고, 한국어 형태소 분석과 공백 토큰화를 비교합니다.